<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT</p>
  <h1 style='font-size:38px;margin:0 0 8px 0;font-weight:900;'>MEETINGS 4 &amp; 5</h1>
  <h2 style='font-size:23px;font-weight:300;margin:0 0 30px 0;color:#D0D7E3;'>Math Essentials + Supervised Machine Learning</h2>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:15px;color:#D0D7E3;margin:0 0 6px 0;'>⏱️ Duration: 2 Hours &nbsp;|&nbsp; 📍 Teaching + Live Coding + Mini Tasks + Project</p>
  <p style='font-size:13px;color:#6B8A9A;margin:0;'>Open in Jupyter Notebook or Google Colab</p>
</div>


---

## 🗺️ Session Roadmap

| Part | Topic | Time |
|------|-------|------|
| **A** | Math Essentials (Summary) — vectors, stats, probability, loss | 25 min |
| **B** | What is Supervised Learning? | 10 min |
| **C** | Linear Regression | 20 min |
| **D** | Logistic Regression | 20 min |
| **E** | Decision Trees & Random Forests | 20 min |
| **F** | Evaluation Metrics | 15 min |
| **G** | Mini Project — Train & Compare 3 Models | 20 min |

> 📖 Read **Markdown cells** for theory. ▶️ **Run code cells**. ✏️ Complete **YOUR TURN** cells.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,
                      'axes.grid':True,'grid.alpha':0.3})
sns.set_palette('muted')
np.random.seed(42)
print('✅  Ready to go.')


---

# 🧮 Part A: Math Essentials — The Quick Version
#### *Just enough intuition to understand what models are doing*

---

We are **not** doing a deep math course. The goal here is intuition — enough that  
formulas you meet later don't feel like magic. Three areas matter: **vectors**, **statistics**, and **the idea of loss/optimisation**.

### 1. Vectors & Features — Linear Algebra in One Idea

Every row of data you feed a model is just a **list of numbers** — a vector.  
A house with `[3 bedrooms, 120 sqm, 5 years old]` is the vector `[3, 120, 5]`.  
"Training a model" really just means finding the right **weights** to combine those numbers into a prediction:

$$\hat{y} = w_1x_1 + w_2x_2 + w_3x_3 + b$$

That's it. Almost everything in classical ML builds on this one equation.

---

### 2. Statistics — Describing Data in Numbers

| Concept | What It Tells You |
|---------|-------------------|
| **Mean** | The average — central tendency |
| **Standard deviation** | How spread out the values are |
| **Correlation** | How strongly two variables move together (-1 to +1) |
| **Probability** | The likelihood of an outcome (0 to 1) — the language models use to express confidence |

---

### 3. Loss & Optimisation — How a Model 'Learns'

A model starts with random guesses. A **loss function** measures how wrong it is.  
Training = repeatedly adjusting weights to make the loss smaller. That's the whole idea.

$$\text{Loss} = \frac{1}{n}\sum (y_{true} - y_{pred})^2 \quad \text{(Mean Squared Error)}$$

> 🔑 **The one-sentence summary:** A model is just an equation with adjustable numbers (weights), and training is the process of nudging those numbers until predictions stop being wrong as often.


In [2]:
t = 5
t = t+1
print(f'the value of t is {t}')

the value of t is 6


In [ ]:
# ── See it, don't just read it: correlation + loss in action ────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Correlation visual
x = np.random.normal(50, 10, 100)
y_pos = x * 1.5 + np.random.normal(0, 8, 100)
axes[0].scatter(x, y_pos, alpha=0.6, color='#0D7377')
axes[0].set_title(f'Positive correlation  (r = {np.corrcoef(x,y_pos)[0,1]:.2f})')
axes[0].set_xlabel('Feature X'); axes[0].set_ylabel('Feature Y')

# Loss surface visual (simple 1D)
w = np.linspace(-2, 4, 100)
loss = (w - 1)**2 + 1
axes[1].plot(w, loss, color='#F0A500', linewidth=2)
axes[1].scatter([1], [1], color='red', zorder=5, label='Best weight (min loss)')
axes[1].set_title('Loss curve — training finds the bottom')
axes[1].set_xlabel('Weight value'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout(); plt.show()
print('Left: correlated features (this is what models exploit to predict)')
print('Right: training = sliding down this curve to the lowest point')


---

# 🎯 Part B: What is Supervised Learning?
#### *Learning from labelled examples*

---

**Supervised learning** means the model learns from data that already has the correct answers (labels).  
You show it many examples of inputs *and* their known outputs, and it learns the mapping between them.

| Type | Predicts | Example |
|------|---------|---------|
| **Regression** | A continuous number | House price, salary, temperature |
| **Classification** | A category | Spam/Not spam, Default/No default |

Today we cover the three most important supervised algorithms: **Linear Regression**, **Logistic Regression**, and **Decision Trees / Random Forests**.


In [ ]:
# ── Our working dataset for this session ────────────────────────────────────
from sklearn.model_selection import train_test_split

n = 300
experience = np.random.uniform(0, 20, n)
education  = np.random.choice([1,2,3,4], n)   # 1=HS 2=Bachelor 3=Master 4=PhD
salary     = 28000 + experience*2800 + education*4000 + np.random.normal(0, 6000, n)

# Binary target for classification later: high earner or not
high_earner = (salary > salary.mean()).astype(int)

df = pd.DataFrame({'experience': experience.round(1), 'education': education,
                    'salary': salary.round(0), 'high_earner': high_earner})
print(df.head())
print(f'\nShape: {df.shape}  |  High earner rate: {df.high_earner.mean():.0%}')


---

# 📈 Part C: Linear Regression
#### *Predicting a continuous number*

---

Linear Regression fits a straight line (or plane, in higher dimensions) through your data,  
minimising the squared distance between predictions and actual values.

$$\hat{y} = w_1 \cdot \text{experience} + w_2 \cdot \text{education} + b$$

**Use it when:** your target is a number (price, salary, temperature) and the relationship is roughly linear.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df[['experience', 'education']]
y = df['salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
preds = lin_reg.predict(X_test)

print('Learned equation:')
print(f'  salary = {lin_reg.coef_[0]:.0f}×experience + {lin_reg.coef_[1]:.0f}×education + {lin_reg.intercept_:.0f}')
print(f'\nMAE:  ${mean_absolute_error(y_test, preds):,.0f}  (avg prediction error)')
print(f'R²:   {r2_score(y_test, preds):.3f}  (1.0 = perfect fit)')

plt.figure(figsize=(6,4))
plt.scatter(y_test, preds, alpha=0.6, color='#0D7377')
plt.plot([y.min(),y.max()],[y.min(),y.max()],'r--', label='Perfect prediction')
plt.xlabel('Actual salary'); plt.ylabel('Predicted salary'); plt.legend()
plt.title('Linear Regression: Predicted vs Actual')
plt.tight_layout(); plt.show()


---

# 🔀 Part D: Logistic Regression
#### *Predicting a category (despite the name 'regression')*

---

Logistic Regression predicts a **probability** between 0 and 1, then applies a threshold (usually 0.5)  
to classify into a category. It squashes the linear equation through an S-shaped curve (sigmoid):

$$P(y=1) = \frac{1}{1 + e^{-(w_1x_1 + w_2x_2 + b)}}$$

**Use it when:** your target is binary (Yes/No, Default/No Default, Spam/Not Spam).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

X = df[['experience', 'education']]
y = df['high_earner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
preds = log_reg.predict(X_test)
probs = log_reg.predict_proba(X_test)[:,1]

print(f'Accuracy: {accuracy_score(y_test, preds):.2%}')
print(f'\nSample predictions (probability → class):')
for p, c in list(zip(probs[:5], preds[:5])):
    print(f'  P(high earner) = {p:.2f}  →  Predicted: {"High earner" if c==1 else "Not high earner"}')

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(4,3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred: No','Pred: Yes'], yticklabels=['Actual: No','Actual: Yes'])
plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()


---

# 🌳 Part E: Decision Trees & Random Forests
#### *Learning by asking a series of yes/no questions*

---

A **Decision Tree** splits data repeatedly using simple rules ("is experience > 10?")  
until it reaches a confident prediction. It's intuitive and visual — but prone to overfitting.

A **Random Forest** trains *many* trees on random subsets of data and features, then averages  
their votes. This drastically reduces overfitting and is one of the most reliable algorithms in practice.

**Use them when:** you want strong performance with minimal tuning, and your data has non-linear patterns.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))

forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
forest.fit(X_train, y_train)
forest_acc = accuracy_score(y_test, forest.predict(X_test))

print(f'Decision Tree accuracy:  {tree_acc:.2%}')
print(f'Random Forest accuracy:  {forest_acc:.2%}')

plt.figure(figsize=(10,5))
plot_tree(tree, feature_names=['experience','education'],
          class_names=['Not High','High'], filled=True, fontsize=9)
plt.title('A Single Decision Tree (max depth 3)')
plt.show()

# Feature importance — a key benefit of tree models
importances = pd.Series(forest.feature_importances_, index=['experience','education'])
print('\nFeature importance (Random Forest):')
print(importances.sort_values(ascending=False).round(3).to_string())


---

## ⏱️ Mini Task 1 — *Predict on New Employees*

> **Time: 10 minutes.** Work solo or in pairs. Run, interpret, be ready to share.

---


Using `log_reg` and `forest` (already trained), predict whether 3 new employees are high earners:

| Employee | Experience | Education |
|----------|-----------|-----------|
| A | 2 years | Bachelor (2) |
| B | 15 years | Master (3) |
| C | 8 years | PhD (4) |

Compare what each model predicts. Do they agree?


In [ ]:
# ✏️  YOUR TURN — Write your solution below

# new_employees = pd.DataFrame({
#     'experience': [2, 15, 8],
#     'education':  [2, 3, 4]
# })
# 
# # Predict with both models and compare
# log_preds = log_reg.predict(new_employees)
# forest_preds = forest.predict(new_employees)
# print(log_preds, forest_preds)

# ── Your code starts here ────────────────────────────────────


---

# 📊 Part F: Evaluation Metrics
#### *How do you know if a model is actually good?*

---

Accuracy alone can be misleading — especially with imbalanced classes. Know these four:

| Metric | Question It Answers | Formula |
|--------|---------------------|--------|
| **Accuracy** | Overall, how often is the model right? | (TP+TN) / Total |
| **Precision** | Of predicted positives, how many were correct? | TP / (TP+FP) |
| **Recall** | Of actual positives, how many did we catch? | TP / (TP+FN) |
| **F1 Score** | Balance of precision and recall | 2×(P×R)/(P+R) |

> 🔑 Use **recall** when missing a positive is costly (disease detection).  
> Use **precision** when false alarms are costly (spam filtering).


In [ ]:
from sklearn.metrics import classification_report

print('=== Logistic Regression ===')
print(classification_report(y_test, log_reg.predict(X_test), target_names=['Not High','High']))

print('=== Random Forest ===')
print(classification_report(y_test, forest.predict(X_test), target_names=['Not High','High']))


---

# 🚀 Part G: Mini Project — Train & Compare 3 Models
#### *Apply everything to a fresh problem*

---

**Task:** Using the dataset below (loan approval), train and compare Logistic Regression,  
Decision Tree, and Random Forest. Report which performs best and why.

**Steps:**
1. Split into train/test (80/20)
2. Train all 3 models
3. Compare accuracy and F1 score in a table
4. State which model you'd deploy and why


In [ ]:
# ── Mini project dataset ─────────────────────────────────────────────────────
n2 = 250
income       = np.random.uniform(20000, 120000, n2)
credit_score = np.random.uniform(500, 800, n2)
loan_amount  = np.random.uniform(2000, 40000, n2)

approval_prob = (income/120000)*0.4 + (credit_score/800)*0.5 - (loan_amount/40000)*0.2
approved = (approval_prob + np.random.normal(0,0.1,n2) > 0.5).astype(int)

loan_df = pd.DataFrame({'income': income.round(0), 'credit_score': credit_score.round(0),
                         'loan_amount': loan_amount.round(0), 'approved': approved})
print(loan_df.head())
print(f'\nApproval rate: {loan_df.approved.mean():.0%}')


In [ ]:
# ✏️  YOUR TURN — Write your solution below

# X = loan_df[['income','credit_score','loan_amount']]
# y = loan_df['approved']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 
# # 1. Train LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
# 
# # 2. Predict on X_test with each
# 
# # 3. Build a comparison table: model name, accuracy, F1 score
# from sklearn.metrics import f1_score
# results = []
# # results.append({'model': 'Logistic Regression', 'accuracy': ..., 'f1': ...})
# 
# # print(pd.DataFrame(results))
# 
# # 4. Which model would you deploy? Write your answer as a comment.

# ── Your code starts here ────────────────────────────────────


---

## ✅ What You Covered Today

| Concept | Status |
|---------|--------|
| Vectors, statistics, loss & optimisation (intuition) | ✅ |
| Supervised learning: regression vs classification | ✅ |
| Linear Regression | ✅ |
| Logistic Regression | ✅ |
| Decision Trees & Random Forests | ✅ |
| Evaluation metrics: accuracy, precision, recall, F1 | ✅ |
| Mini task + mini project completed | ✅ |

---

## 📚 Before Meeting 6
Complete the mini project above if you haven't finished it. Try changing `max_depth` on the
Random Forest and see how accuracy changes — that's a preview of hyperparameter tuning.

> 🔭 **Next up — Meeting 6:** SVMs, KNN, Naive Bayes, overfitting vs underfitting, cross-validation, and scikit-learn pipelines.

---

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:30px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 8px 0;color:#14BDBD;'>You just trained 3 real ML models.</h3>
  <p style='color:#F0A500;font-weight:bold;margin:10px 0 0 0;'>See you in Meeting 6. 🚀</p>
</div>
